In [33]:
import sys, os
sys.path.append(os.path.join('/home/module'))

from Bq_Connect import BQDataProcessor  
sql = BQDataProcessor(env="dev") 

[BQDataProcessor] Connected → env=DEV, project=pgd-dev-data-analytics, SA=default (/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json)


In [36]:
DEST_TABLE       = "pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_kredit"             
PARTITION_BY     = "tgl_transaksi"                                    

SELECT_QUERY = """
SELECT
    ROW_NUMBER() OVER (ORDER BY t.no_kontrak)              AS rn,
    t.cif,
    t.no_kontrak,
    t.branch_code                                           AS sub_branch_cd,
    vt.sub_branch_nm,
    vt.branch_cd,
    vt.branch_nm,
    vt.area_nm,
    vt.region_cd,
    vt.region_nm,
    cast(coalesce(t.create_date, t.tgl_kredit) as TIMESTAMP) AS tgl_transaksi,    -- sebelumnya menggunakan create_date tapi isinya null semua
    t.product_code,
    t.last_flow_code                                        AS nama_trx,
    t.up,
    CAST(NULL AS STRING)                                    AS transaksi_up,
    t.flag
FROM pgd-prod-data-analytics.datalake.pgd_tbl_kredit t
LEFT JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch vt
    ON t.branch_code = vt.sub_branch_cd
WHERE
    --t.create_date IS NOT NULL                                                  -- sebelumnya ini not null, tapi sekarang semuanya null
    t.status_rek = '4'
"""


In [37]:
sql.execute_ddl(f"""create or replace table {DEST_TABLE} as 
{SELECT_QUERY}
""")

print(sql.read(f"select * from {DEST_TABLE} limit 10"))
print(sql.read(f"select count(*) from {DEST_TABLE} limit 10"))

[DDL] Executing: create or replace table pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_kredit as 

SELECT
    ROW_NUMBER() OVER (O...
[DDL] Success.
[READ] Executing query...
[READ] Success → 10 row(s) returned.
        rn         cif        no_kontrak sub_branch_cd sub_branch_nm  \
0  7076403  1008551703  1283125320010318         12831     CP GROGOL   
1  7076410  1007242178  1283125320010847         12831     CP GROGOL   
2  7076432  9002317622  1283125320012629         12831     CP GROGOL   
3  7076515  1001941592  1283125370002157         12831     CP GROGOL   
4  7076519  1033828451  1283125370002215         12831     CP GROGOL   
5  7076535  1024939480  1283125370002405         12831     CP GROGOL   
6  7076536  1013921924  1283125370002447         12831     CP GROGOL   
7  7076573  1031392729  1283125380000076         12831     CP GROGOL   
8  7076584  1032400234  1283125670000257         12831     CP GROGOL   
9  7076692  1031590335  1283126010001187         12831     CP